# Exploitation notebook (no video recording because of OS)

In [1]:
import gym
import highway_env
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from highway_env.envs.common.evaluate import PrintMetrics


# Visualisationd
# import sys>
# sys.path.append('C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/')     
from ipywidgets import interact, widgets
import glob
import numpy as np

ABS_PATH = 'C:/Users/pigo/Desktop/HighwayDRL'

### Choose the environment to exploit the model in

In [2]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

### Choose the model

In [3]:
model_id = widgets.Text()
def m(input_model):
    model_id.value = str(input_model)
interact(m, input_model=glob.glob(f"{ABS_PATH}/training_output/models/*.zip"))


interactive(children=(Dropdown(description='input_model', options=('C:/Users/pigo/Desktop/HighwayDRL/training_…

<function __main__.m(input_model)>

In [5]:
# Number of exploitation episodes
episode_num = 1
metricObj = PrintMetrics()

# metric counters
left_lane_change, right_lane_change = None, None

env = gym.make(env_id.value)
env.configure({
    "screen_width":1500,
    "screen_height":250
})

model = PPO.load(model_id.value)

for episode in range(episode_num):

    obs, done = env.reset(), False

    # Reset metric counters at the beginning of each episode
    decision_change_num, left_lane_change_num, right_lane_change_num = 0, 0, 0
    accelerations, decelerations, speeds = [], [], []

    while not done:

        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, info = env.step(action)
        env.render()

        # Update metric counters
        decision_change_num += env.DECISION_CHANGE
        speeds.append(env.vehicle.speed)
        if env.vehicle.throttle > 0:
            accelerations.append(env.vehicle.throttle)
        else:
            decelerations.append(env.vehicle.throttle)

        if env.LAST_LANE_IDX != env.vehicle.lane_index[2]:
            if env.LAST_LANE_IDX > env.vehicle.lane_index[2] and env.LAST_LANE_IDX != 1000:
                left_lane_change_num += 1
            else:
                right_lane_change_num += 1
            env.LAST_LANE_IDX = env.vehicle.lane_index[2]

    episode_duration = (env.steps/env.config["policy_frequency"]*env.config["simulation_frequency"])/30
    decision_change_rate = decision_change_num / episode_duration
    left_lane_change_rate = left_lane_change_num / episode_duration
    right_lane_change_rate = right_lane_change_num / episode_duration
    mean_speed = np.mean(speeds)
    km_travelled = (mean_speed * episode_duration)/1000
    mean_acceleration = np.mean(accelerations)
    mean_deceleration = np.mean(decelerations)
    metricObj.printEpisode(km_travelled, decision_change_num, decision_change_rate, left_lane_change_num, left_lane_change_rate,\
        right_lane_change_num, right_lane_change_rate, mean_speed*3.6, mean_acceleration, mean_deceleration, env.vehicle.crashed, episode_duration, episode+1)

env.close()
metricObj.printRecap(episode_num, f"{ABS_PATH}/exploitation/eval_metrics/")


episode 1 ended, metrics:             
	episode duration: 20 seconds,
	collision? YES 
	km travelled: 0.71,             
	decision changes: 23, 
	mean speed: 125.33 km/h, 
	mean acceleration: 0.429 m/s, 
	mean deceleration: -0.5 m/s,             
	decision change rate: 1.12, 
	left lane changes: 3, 
	left lane change rate: 0.15             
	right lane changes: 4, 
	right lane change rate: 0.2

evaluation ended, metrics:            
means: 
	mean episode duration: 40 seconds, 
	mean km travelled: 1.42,            
	overall mean speed: 126.51 km/h, 
	overall mean acceleration: 0.328 m/s, 
	overall mean deceleration: -0.25 m/s            
	mean decision changes: 46, 
	mean decision change rate: 1.13,            
	mean left lane changes: 2, 
	mean left lane change rate: 0.09,            
	mean right lane changes: 4, 
	mean right lane change rate: 0.13,             
total:
	total km travelled: 2.84, 
	total decision changes: 91,            
	total left lane changes: 5, 
	total right lane 